## Run GraphBG (our method)

In [1]:
from numpy.random import default_rng
import scanpy as sc
from anndata import AnnData
import scipy
sc.logging.print_header()
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score
import pickle
import os.path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.9.3 anndata==0.9.2 umap==0.5.7 numpy==1.24.4 scipy==1.8.1 pandas==1.4.2 scikit-learn==1.3.2 statsmodels==0.14.1 python-igraph==0.11.8 pynndescent==0.5.13


In [2]:
adata = sc.read_visium('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data', count_file='filtered_feature_bc_matrix.h5')
ground_truth = pd.read_csv('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/metadata.tsv', delimiter='\t', index_col=0)
adata.obs['ground_truth'] = ground_truth['ground_truth']
adata.var_names_make_unique()

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
adata.var_names_make_unique()
true_labels = pd.factorize(adata.obs['ground_truth'])[0] 
n_clusters = len(np.unique(true_labels))

n_features = adata.shape[1]
if n_features > 5000:
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.scale(adata, zero_center=False, max_value=10)
if n_features > 5000:
    adata =  adata[:, adata.var['highly_variable']]
# Perform PCA
sc.tl.pca(adata, n_comps=50)
adata.obsm['X_pca'].shape
if scipy.sparse.issparse(adata.X):
    adata.obsm['X_feature'] = adata.X.todense() 
else:
    adata.obsm['X_feature'] = adata.X
adata.obsm["X_feature"] = np.asarray(adata.obsm["X_feature"])
pca = PCA(n_components=20, random_state=42)
adata.obsm['X_pca'] = pca.fit_transform(np.asarray(adata.obsm['X_feature']))

In [4]:
from utils import runGraphBG
runGraphBG(adata, n_clusters=n_clusters, radius=50, key='X_pca', refinement=True, n_neighbors = 4, n_components = 20, covariance_type = 'full', n_init = 5, max_iter = 2000)

full ward
Data input shape n:  (3798, 20)


In [5]:
adata.obs["GraphBG"] = adata.obs["domain"].copy()

In [6]:
NMI = normalized_mutual_info_score(adata.obs['GraphBG'], adata.obs['ground_truth'])
print(NMI)

0.703806177050863


In [7]:
adata.write_h5ad(
    '/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/Section_1_GraphBG.h5ad'
)